<a href="https://colab.research.google.com/github/CathalD/NorthStarProject_CoastalBlueCarbonMMRV/blob/main/CoastalBlueCarbon_LargeScaleCovariateExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='north-star-project-470316')

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
# ============================================================================
# 2. LOAD AND FILTER DATA
# ============================================================================
# Input:  global_cores_harmonized_VM0033.csv  (output of 03_Janousek_harmonize.R)
# Output: coastal BC blue carbon cores (EM + SG ecosystems, lat 46.5–55.5°N)
#
# Ecosystem codes retained:
#   EM = Estuarine Marsh (saltmarsh)  ← blue carbon per VM0033
#   SG = Seagrass                     ← blue carbon per VM0033
#
# Ecosystem codes REMOVED:
#   FL = Freshwater / freshwater marsh ← NOT a blue carbon ecosystem under VM0033
#        (freshwater wetlands have different decomposition dynamics and are
#         not eligible for tidal wetland restoration crediting)
#
# Geographic filter: lat 46.5–55.5°N = British Columbia coast (excludes Alaska)
# ============================================================================

FILENAME = 'global_cores_harmonized_VM0033.csv'
try:
    df = pd.read_csv(FILENAME)
except FileNotFoundError:
    print("❌ ERROR: File not found. Upload global_cores_harmonized_VM0033.csv")
    raise

# Blue carbon ecosystem types only (no FL freshwater)
target_ecosystems = ['EM', 'SG']   # ← removed 'FL'
MIN_LAT, MAX_LAT = 46.5, 55.5

df_subset = df[
    (df['ecosystem'].isin(target_ecosystems)) &
    (df['latitude'] >= MIN_LAT) &
    (df['latitude'] <= MAX_LAT)
].copy()

# Prepare GEE Features (one per unique core location)
unique_locs = (
    df_subset[['core_id', 'longitude', 'latitude']]
    .drop_duplicates(subset='core_id')
    .dropna()
)
feature_list = []
for _, row in unique_locs.iterrows():
    geom = ee.Geometry.Point([row['longitude'], row['latitude']])
    feature_list.append(ee.Feature(geom, {'core_id': str(row['core_id'])}))

print(f"✓ Ecosystems retained: {target_ecosystems} (VM0033 blue carbon only; FL excluded)")
print(f"✓ Lat filter: {MIN_LAT}°–{MAX_LAT}°N (BC coast)")
print(f"✓ Processing {len(feature_list)} unique core locations")
print(f"  Ecosystem breakdown: {df_subset['ecosystem'].value_counts().to_dict()}")

In [ ]:
# ============================================================================
# 3. DEFINE CANONICAL COVARIATE STACKS
# ============================================================================
# Canonical 27-band stack — MUST match GoogleEarthEngineAOICovariateAnalysis.js
# and CoastalBlueCarbon_GlobalCoreCovariate_Extraction.ipynb exactly.
#
# Group 1 — Topography & Channels (7):
#   elevation_m, slope, elevationRelMHW, twi, dist_to_channel_m,
#   tidal_flat_prob, coastal_dist_m
# Group 2 — Sentinel-1 SAR (3):
#   VV_mean, VH_mean, VVVH_ratio
# Group 3 — Sentinel-2 Optical & Phenology (15):
#   B, G, R, B5, B6, B7, NIR, SWIR1, SWIR2
#   NDVI_median, LSWI_median, mNDWI_median, NDVI_stdDev, SAVI_median, tidal_wetness
# Group 4 — Climate (2):
#   MAT_C, MAP_mm
#
# NOTE on SoilGrids:
#   stack_sg is defined below for use as a Bayesian prior in the R workflow
#   (P3_0c_bayesian_prior_setup_bluecarbon.R). It is NOT part of the canonical
#   27-band covariate stack and is NOT extracted at core point locations.
#   Red-Edge bands (B5, B6, B7) replace the three former SoilGrids covariate
#   slots — they are primary chlorophyll-sensitive predictors for Zostera
#   (eelgrass) and Spartina (saltmarsh) carbon burial rates.
# ============================================================================

# ── DATE RANGES (match GEE JS script) ────────────────────────────
S2_START  = '2020-01-01'
S2_END    = '2023-12-31'
SAR_START = '2020-01-01'
SAR_END   = '2023-12-31'
TC_START  = '2000-01-01'
TC_END    = '2022-12-31'
S2_CLOUD_THRESHOLD = 20


# ── GROUP 1: Topography & Channels ───────────────────────────────
dem         = ee.Image('NASA/NASADEM_HGT/001').select('elevation')
elevation_m = dem.rename('elevation_m')
slope       = ee.Terrain.slope(dem).rename('slope')

# Elevation relative to Mean High Water (MHW)
# Approximation: DEM - 0.5 m proxy for MHW above MSL in BC coastal areas.
elevRelMHW = dem.subtract(0.5).rename('elevationRelMHW')

# TWI — Topographic Wetness Index = ln(upslope_area / tan(slope))
# High TWI = flat/concave terrain prone to waterlogging (marsh hollows, channels)
def compute_twi(dem_img):
    slope_rad = ee.Terrain.slope(dem_img).multiply(3.14159 / 180)
    tan_slope = slope_rad.tan().max(0.001)
    contrib   = dem_img.gte(-9999).unmask(0).reduceNeighborhood(
        reducer=ee.Reducer.sum(),
        kernel=ee.Kernel.circle(**{'radius': 20, 'units': 'pixels'})
    ).max(1)
    return contrib.divide(tan_slope).log().rename('twi')

twi = compute_twi(dem)

# dist_to_channel_m — distance to tidal channels (replaces TPI, v1.4/v1.7)
# JRC Global Surface Water occurrence > 30% as channel mask.
# focalMax(30 m) closes canopy-gap artefacts; fastDistanceTransform × 30 = metres.
channel_mask = (ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
                .select('occurrence').gt(30).unmask(0)
                .focalMax(30, 'circle', 'meters'))
dist_to_channel = (
    channel_mask
    .fastDistanceTransform(500, 'pixels', 'squared_euclidean')
    .sqrt()
    .multiply(30)
    .rename('dist_to_channel_m').float()
)

# Tidal flat probability — Murray et al. 2019 Global Intertidal Change
# FIX v1.8: Use ee.ImageCollection(...).filterBounds(...).mosaic() — NOT ee.Image(...)
# to avoid asset-type error and global-scale timeout.
try:
    tidal_flat_prob = (
        ee.ImageCollection('UQ/murray/Intertidal/v1_1/global_intertidal')
        .filterBounds(ee.Geometry.BBox(-140, 46, -120, 56))  # BC coast bounding box
        .mosaic()
        .select('classification').eq(1).unmask(0)
        .rename('tidal_flat_prob').float()
    )
    print('✓ Murray intertidal dataset loaded (ImageCollection mosaic, BC bbox)')
except Exception:
    tidal_flat_prob = ee.Image(0).rename('tidal_flat_prob').float()
    print('⚠ Murray intertidal dataset unavailable — tidal_flat_prob set to 0')

# Coastal distance — JRC Global Surface Water occurrence > 50% as shoreline proxy
water_mask  = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').gt(50).unmask(0)
coastal_dist = (
    water_mask.fastDistanceTransform(500, 'pixels', 'squared_euclidean')
    .sqrt().multiply(30).rename('coastal_dist_m').float()
)

stack_topo = (elevation_m.addBands(slope).addBands(elevRelMHW)
              .addBands(twi).addBands(dist_to_channel)
              .addBands(tidal_flat_prob).addBands(coastal_dist))
print('✓ Group 1 — Topography & Channels (7):')
print('    elevation_m, slope, elevationRelMHW, twi, dist_to_channel_m,')
print('    tidal_flat_prob, coastal_dist_m')
print('    (tpi removed; dist_to_channel_m added — more direct hydrological driver)')


# ── GROUP 2: Sentinel-1 SAR ───────────────────────────────────────
# FIX v1.6: Noise filter applied per-pixel via .map(), not as collection metadata filter.
s1_col = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterDate(SAR_START, SAR_END)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .map(lambda img: img.updateMask(img.select('VV').gt(-30)))   # pixel-level noise mask
)
s1_med = s1_col.median()
vv     = s1_med.select('VV').rename('VV_mean')
vh     = s1_med.select('VH').rename('VH_mean')
vvvh   = s1_med.select('VV').subtract(s1_med.select('VH')).rename('VVVH_ratio')
stack_s1 = vv.addBands(vh).addBands(vvvh)
print('✓ Group 2 — SAR (3): VV_mean, VH_mean, VVVH_ratio')


# ── GROUP 3: Sentinel-2 Optical & Phenology ───────────────────────
# Summer months (May–Sep): peak tidal marsh biomass in BC.
# FIX v1.8: .filterBounds() added — scopes collection to BC coastal tiles
# to prevent global-scene timeout at render time.
#
# CHANGES vs previous version:
#   - EVI_median REMOVED (replaced by NDVI_stdDev + Red-Edge chlorophyll bands)
#   - B5, B6, B7 (Red-Edge 705/740/783 nm) ADDED — primary chlorophyll proxies
#     for Zostera (eelgrass) and Spartina (saltmarsh) carbon burial rates
#   - NDVI_stdDev ADDED — phenological variability (computed before median)
#   - NDBI removed (previous version); TC brightness + greenness removed (previous version)

def process_s2_image(image):
    qa        = image.select('QA60')
    cloud_mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    ndwi_mask  = image.normalizedDifference(['B3', 'B8']).lt(0.1)
    return image.updateMask(cloud_mask.And(ndwi_mask)).divide(10000)

# BC coast bounding box — scopes collection to relevant tiles
BC_BBOX = ee.Geometry.BBox(-140, 46, -120, 56)

s2_coll = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate(S2_START, S2_END)
    .filterBounds(BC_BBOX)   # FIX v1.8: scope to BC coastal tiles
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', S2_CLOUD_THRESHOLD))
    .filter(ee.Filter.calendarRange(5, 9, 'month'))   # May–September
    .limit(500)
    .map(process_s2_image)
)

# NDVI_stdDev — computed over full summer collection BEFORE median.
# High = seasonally dynamic (tidal marsh). Low = permanent water or stable veg.
# FIX v1.7: .select() restricts stdDev pass to one band, reducing memory use.
ndvi_stddev = (s2_coll
    .select(s2_coll.first().normalizedDifference(['B8', 'B4']).rename('NDVI_stdDev').bandNames())
    .reduce(ee.Reducer.stdDev())
    .rename('NDVI_stdDev'))
# Simpler approach that avoids band name issue:
ndvi_stddev = (
    s2_coll
    .map(lambda img: img.normalizedDifference(['B8', 'B4']).rename('NDVI_stdDev'))
    .reduce(ee.Reducer.stdDev())
    .rename('NDVI_stdDev')
)

s2 = s2_coll.median()

# Raw reflectance bands — includes Red-Edge B5, B6, B7
B     = s2.select('B2').rename('B')
G     = s2.select('B3').rename('G')
R     = s2.select('B4').rename('R')
B5    = s2.select('B5').rename('B5')   # Red-Edge 705 nm
B6    = s2.select('B6').rename('B6')   # Red-Edge 740 nm
B7    = s2.select('B7').rename('B7')   # Red-Edge 783 nm
NIR   = s2.select('B8').rename('NIR')
SWIR1 = s2.select('B11').rename('SWIR1')
SWIR2 = s2.select('B12').rename('SWIR2')

# Spectral indices — coastal blue carbon relevant
ndvi  = s2.normalizedDifference(['B8', 'B4']).rename('NDVI_median')
lswi  = s2.normalizedDifference(['B8', 'B11']).rename('LSWI_median')
mndwi = s2.normalizedDifference(['B3', 'B11']).rename('mNDWI_median')
savi  = s2.expression('1.5*(NIR-RED)/(NIR+RED+0.5)',
                       {'NIR': NIR, 'RED': R}).rename('SAVI_median')
# EVI removed — replaced by NDVI_stdDev and Red-Edge bands

# Tasseled Cap Wetness ONLY (Nedkov 2017 Sentinel-2 SR coefficients)
# Most inundation-sensitive TC component; brightness + greenness are forest metrics.
tidal_wetness = s2.expression(
    '0.1511*B + 0.1973*G + 0.3283*R + 0.3407*NIR + (-0.7117)*SWIR1 + (-0.4559)*SWIR2',
    {'B': B, 'G': G, 'R': R, 'NIR': NIR, 'SWIR1': SWIR1, 'SWIR2': SWIR2}
).rename('tidal_wetness')

# Stack in canonical order: 9 raw + 6 derived = 15 S2 bands
stack_s2 = (B.addBands(G).addBands(R).addBands(B5).addBands(B6).addBands(B7)
            .addBands(NIR).addBands(SWIR1).addBands(SWIR2)
            .addBands(ndvi).addBands(lswi).addBands(mndwi)
            .addBands(ndvi_stddev).addBands(savi).addBands(tidal_wetness))
print('✓ Group 3 — Sentinel-2 Optical & Phenology (15):')
print('    Raw (9): B, G, R, B5, B6, B7, NIR, SWIR1, SWIR2')
print('    Derived (6): NDVI_median, LSWI_median, mNDWI_median, NDVI_stdDev, SAVI_median, tidal_wetness')
print('    (EVI removed; B5/B6/B7 Red-Edge added; NDVI_stdDev added as phenology metric)')


# ── GROUP 4: TerraClimate (MAT + MAP) ─────────────────────────────
tc_col   = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').filterDate(TC_START, TC_END)
tc_mean  = tc_col.mean()
mat      = tc_mean.expression('((tmmn + tmmx) / 2.0) / 10.0',
                               {'tmmn': tc_mean.select('tmmn'),
                                'tmmx': tc_mean.select('tmmx')}).rename('MAT_C')
map_mm   = tc_mean.select('pr').multiply(12).rename('MAP_mm')
stack_climate = mat.addBands(map_mm)
print(f'✓ Group 4 — Climate (2): MAT_C (°C), MAP_mm (mm/yr) — TerraClimate {TC_START[:4]}–{TC_END[:4]}')


# ── SoilGrids SOC — BAYESIAN PRIOR ONLY (NOT a canonical covariate) ──────────
# stack_sg is defined for reference and the Bayesian R prior workflow.
# It is NOT extracted at core point locations and NOT in CANONICAL_BANDS.
# Correct band names: soc_0-5cm_mean, soc_5-15cm_mean, etc. (not 'ocs_*').
sg      = ee.Image('projects/soilgrids-isric/soc_mean')
sg_bdod = ee.Image('projects/soilgrids-isric/bdod_mean')
def _ocs_layer(soc_band, bd_band, thickness):
    return (sg.select(soc_band).divide(10)
            .multiply(sg_bdod.select(bd_band).divide(100))
            .multiply(thickness).divide(100))
sg_0_30   = (_ocs_layer('soc_0-5cm_mean', 'bdod_0-5cm_mean', 5)
             .add(_ocs_layer('soc_5-15cm_mean', 'bdod_5-15cm_mean', 10))
             .add(_ocs_layer('soc_15-30cm_mean', 'bdod_15-30cm_mean', 15))
             .rename('sg_soc_0_30cm'))
sg_30_100 = (_ocs_layer('soc_30-60cm_mean', 'bdod_30-60cm_mean', 30)
             .add(_ocs_layer('soc_60-100cm_mean', 'bdod_60-100cm_mean', 40))
             .rename('sg_soc_30_100cm'))
sg_0_100  = sg_0_30.add(sg_30_100).rename('sg_soc_0_100cm')
stack_sg  = sg_0_30.addBands(sg_30_100).addBands(sg_0_100)
print('✓ SoilGrids SOC prior defined (Bayesian R workflow only — NOT a canonical covariate):')
print('    sg_soc_0_30cm, sg_soc_30_100cm, sg_soc_0_100cm')
print()
print('═══════════════════════════════════════════════════════')
print('Canonical 27-band stack (v1.7/v1.8):')
print('  7 topo/channels + 3 SAR + 15 Sentinel-2 + 2 climate = 27')
print('  SoilGrids: external Bayesian prior only (NOT in canonical stack)')


In [ ]:
# ============================================================================
# 4. BATCH EXTRACTION — CANONICAL COVARIATE GROUPS
# ============================================================================
def extract_batch(image, features, name, batch_size=50, scale=30):
    """Extract pixel values at point locations using batched reduceRegions."""
    results = []
    n_batches = (len(features) + batch_size - 1) // batch_size
    n_failed  = 0
    print(f"--- Extracting {name} ({len(features)} pts, batch={batch_size}, scale={scale}m) ---")
    for i in range(0, len(features), batch_size):
        batch     = features[i: i + batch_size]
        fc        = ee.FeatureCollection(batch)
        batch_num = i // batch_size + 1
        try:
            res  = image.reduceRegions(collection=fc, reducer=ee.Reducer.first(),
                                       scale=scale, tileScale=2)
            data = res.getInfo()['features']
            for f in data:
                results.append(f['properties'])
            if batch_num % 5 == 0 or batch_num == n_batches:
                print(f"   Batch {batch_num}/{n_batches} OK  ({len(results)} rows)")
        except Exception as e:
            n_failed += 1
            print(f"   Batch {batch_num}/{n_batches} FAILED: {e}")
    print(f"✓ {name} — {len(results)} rows, {n_failed} batches failed")
    return pd.DataFrame(results)

# Group 1: Topography & Channels (7 bands — static global layers, large batch OK)
df_topo = extract_batch(stack_topo, feature_list, "Topography & Channels", batch_size=500, scale=30)

# Group 2: Sentinel-1 SAR (3 bands — 2020–2023 median)
df_s1   = extract_batch(stack_s1,   feature_list, "Sentinel-1 SAR",       batch_size=100, scale=30)

# Group 3: Sentinel-2 optical + phenology (15 bands — S2 median + NDVI_stdDev)
# Smaller batch size: 15-band stack is memory-intensive in GEE
df_s2   = extract_batch(stack_s2,   feature_list, "Sentinel-2 Optical & Phenology", batch_size=50, scale=30)

# Group 4: TerraClimate climate variables (2 bands — ~4 km native resolution)
df_clim = extract_batch(stack_climate, feature_list, "TerraClimate Climate", batch_size=500, scale=4000)

# NOTE: SoilGrids (stack_sg) is NOT extracted here.
# It is a Bayesian prior for the R workflow only (P3_0c_bayesian_prior_setup_bluecarbon.R).
# sg_soc_* bands are NOT part of the canonical 27-band covariate stack.

print("\n=== Extraction complete ===")
print(f"  Topography & Channels: {len(df_topo)} rows | {len(df_topo.columns)} cols  (7 bands)")
print(f"  Sentinel-1 SAR:        {len(df_s1)} rows | {len(df_s1.columns)} cols  (3 bands)")
print(f"  Sentinel-2 + Phenology:{len(df_s2)} rows | {len(df_s2.columns)} cols  (15 bands: 9 raw + 6 derived incl. NDVI_stdDev)")
print(f"  TerraClimate:          {len(df_clim)} rows | {len(df_clim.columns)} cols  (2 bands)")
print(f"  Total canonical bands: 7 + 3 + 15 + 2 = 27")

In [ ]:
# ============================================================================
# 5. MERGE ALL COVARIATE GROUPS + EXPORT
# ============================================================================
# Output column order follows the canonical 27-band sequence exactly,
# matching GoogleEarthEngineAOICovariateAnalysis.js (Step ⑥ band_names array)
# and CoastalBlueCarbon_GlobalCoreCovariate_Extraction.ipynb (CANONICAL_BANDS).
#
# Group 1 — Topography & Channels (7):
#   elevation_m, slope, elevationRelMHW, twi, dist_to_channel_m,
#   tidal_flat_prob, coastal_dist_m
# Group 2 — Sentinel-1 SAR (3):
#   VV_mean, VH_mean, VVVH_ratio
# Group 3 — Sentinel-2 Optical & Phenology (15):
#   B, G, R, B5, B6, B7, NIR, SWIR1, SWIR2,
#   NDVI_median, LSWI_median, mNDWI_median, NDVI_stdDev, SAVI_median, tidal_wetness
# Group 4 — Climate (2):
#   MAT_C, MAP_mm
#
# TOTAL: 27 canonical bands
#
# NOTE: SoilGrids (sg_soc_*) is NOT merged here — it is a Bayesian prior
# used by P3_0c_bayesian_prior_setup_bluecarbon.R, not a model covariate.
# ============================================================================

def merge_df(main, sub, key='core_id'):
    if sub is None or sub.empty:
        return main
    drop_cols = ['system:index']
    keep_cols = [key] + [c for c in sub.columns if c not in [key] + drop_cols]
    return pd.merge(main, sub[keep_cols], on=key, how='left')

final = df_subset.copy()
final['core_id'] = final['core_id'].astype(str)

for sub_df in [df_topo, df_s1, df_s2, df_clim]:
    if not sub_df.empty and 'core_id' in sub_df.columns:
        sub_df['core_id'] = sub_df['core_id'].astype(str)
    final = merge_df(final, sub_df)

# ── Canonical 27-band column order (v1.7/v1.8) ─────────────────────
CANONICAL_BANDS = [
    # Group 1 — Topography & Channels (7)
    'elevation_m', 'slope', 'elevationRelMHW', 'twi', 'dist_to_channel_m',
    'tidal_flat_prob', 'coastal_dist_m',
    # Group 2 — Sentinel-1 SAR (3)
    'VV_mean', 'VH_mean', 'VVVH_ratio',
    # Group 3 — Sentinel-2 Optical & Phenology (15: 9 raw + 6 derived)
    'B', 'G', 'R', 'B5', 'B6', 'B7', 'NIR', 'SWIR1', 'SWIR2',
    'NDVI_median', 'LSWI_median', 'mNDWI_median',
    'NDVI_stdDev', 'SAVI_median', 'tidal_wetness',
    # Group 4 — Climate (2)
    'MAT_C', 'MAP_mm',
]
assert len(CANONICAL_BANDS) == 27, f"Expected 27 canonical bands, got {len(CANONICAL_BANDS)}"

# Confirm removed bands are absent
_removed_check = {'tpi', 'EVI_median', 'sg_soc_0_30cm', 'sg_soc_30_100cm', 'sg_soc_0_100cm'}
assert not _removed_check.intersection(CANONICAL_BANDS), \
    f"Removed bands found in CANONICAL_BANDS: {_removed_check.intersection(CANONICAL_BANDS)}"

meta_cols     = [c for c in final.columns if c not in CANONICAL_BANDS]
present_bands = [b for b in CANONICAL_BANDS if b in final.columns]
missing_bands = [b for b in CANONICAL_BANDS if b not in final.columns]

final = final[meta_cols + present_bands]

print(f"Final dataset: {len(final)} rows × {len(final.columns)} cols")
print(f"  Meta columns:      {len(meta_cols)}")
print(f"  Canonical bands:   {len(present_bands)}/27 present")
if missing_bands:
    print(f"  ⚠ Missing bands:   {missing_bands}")
else:
    print(f"  ✓ All 27 canonical bands present")

print(f"\nEcosystem breakdown:\n{final['ecosystem'].value_counts().to_string()}")
print(f"\nNA rates for key covariates:")
for b in ['elevation_m', 'dist_to_channel_m', 'NDVI_median', 'NDVI_stdDev',
          'B5', 'VV_mean', 'MAT_C']:
    if b in final.columns:
        na_pct = final[b].isna().mean() * 100
        print(f"  {b}: {na_pct:.1f}% missing")

# ── Save output ─────────────────────────────────────────────────────
OUTFILE = 'global_cores_with_gee_covariates_BC.csv'
final.to_csv(OUTFILE, index=False)
files.download(OUTFILE)

print(f"\n✓ Saved: {OUTFILE}")
print(f"  → {len(final)} BC coastal cores (EM + SG only, VM0033) with 27 canonical covariates")
print(f"  → Column order matches GEE JS script (band_names array, Step ⑥)")
print(f"  → Copy to: BlueCarbon_Workflow_V1.0/Pre-Analysis Data Preparation/data_global/")
print(f"  → Used by: P4_3b_Depth_Harmonization_Global.R + P4_05_Transfer_Learning_Modelling.R")